# Gold Layer — `fact_accrual_worklist`

**Source:** `[pe]_xx102_pe_finance.silver.accrual_worklist`

**Purpose:** Curated fact table with essential columns for downstream reporting, dashboards, and analytics consumers.

**Outputs:**
1. Delta table: `[pe]_xx102_pe_finance.gold.fact_accrual_worklist`
2. CSV export: `/Volumes/[pe]_xx102_pe_finance/gold/accrual_matching/output/fact_accrual_worklist.csv`

**Columns Selected (10):**
| Column | Description |
|--------|-------------|
| CompanyCode | SAP Company Code |
| CompanyCodeName | Company Code description |
| PurchaseOrder | PO number |
| Supplier | Vendor/Supplier ID |
| GLAccount | General Ledger account |
| Exposure_Amount | GR minus IR (open accrual) in CC currency |
| CompanyCodeCurrency | Local currency of the company code |
| Oldest_GR_Date | Earliest GR posting date |
| Days_Open | Days since oldest GR |
| Recommended_Action | POST / REVIEW / OBSERVE / NO ACTION |

In [0]:
# ── Configuration ────────────────────────────────────────────────────────────
CATALOG = "[pe]_xx102_pe_finance"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"
SOURCE_TABLE = "accrual_worklist"
TARGET_TABLE = "fact_accrual_worklist"
OUTPUT_PATH = f"/Volumes/{CATALOG}/{SCHEMA_GOLD}/accrual_matching/output"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{SCHEMA_GOLD}`")

print(f"Source: {CATALOG}.{SCHEMA_SILVER}.{SOURCE_TABLE}")
print(f"Target: {CATALOG}.{SCHEMA_GOLD}.{TARGET_TABLE}")
print(f"CSV Output: {OUTPUT_PATH}/")

Source: [pe]_xx102_pe_finance.silver.accrual_worklist
Target: [pe]_xx102_pe_finance.gold.fact_accrual_worklist
CSV Output: /Volumes/[pe]_xx102_pe_finance/gold/accrual_matching/output/


In [0]:
from pyspark.sql import functions as F

# ── Step 1: Read Silver Accrual Worklist & Select Curated Columns ────────────
df_silver = spark.table(f"`{CATALOG}`.`{SCHEMA_SILVER}`.{SOURCE_TABLE}")

# Select only downstream-relevant columns for the fact table
df_fact = (
    df_silver
    .select(
        F.col("CompanyCode"),
        F.col("CompanyCodeName"),
        F.col("PurchaseOrder").cast("long").alias("PurchaseOrder"),
        F.col("Supplier"),
        F.col("GLAccount"),
        F.col("Exposure_Amount"),
        F.col("Exposure_Amount_USD"),
        F.col("CompanyCodeCurrency"),
        F.col("Oldest_GR_Date").cast("date").alias("Oldest_GR_Date"),
        F.col("Days_Open"),
        F.col("Fiscal_Year"),
        F.col("Fiscal_Period"),
        F.col("Worklist_Category"),
        F.col("RCA"),
        F.col("Recommended_Action"),
    )
    .orderBy(F.col("Exposure_Amount").desc())
)

print(f"Fact table rows: {df_fact.count()}")
print(f"Columns: {len(df_fact.columns)}")
df_fact.display()

Fact table rows: 87
Columns: 15


CompanyCode,CompanyCodeName,PurchaseOrder,Supplier,GLAccount,Exposure_Amount,Exposure_Amount_USD,CompanyCodeCurrency,Oldest_GR_Date,Days_Open,Fiscal_Year,Fiscal_Period,Worklist_Category,RCA,Recommended_Action
4010,Company Code HR 4010,4500100156,0050100023,64000000,1296373.2,1400083.06,EUR,2025-04-23,415,2025,4,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2025/04 but IR posted in later period 2025/06. Exposure 1296373.2 EUR requires period-end accrual adjustment.,POST
2210,Company Code CH 2210,4500100051,0050100029,52000000,701842.32,786063.4,CHF,2026-04-02,71,2026,4,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2026/04 but IR posted in later period 2026/05. Exposure 701842.32 CHF requires period-end accrual adjustment.,POST
5010,Company Code EE 5010,4500100097,0050100011,65000000,696458.72,752175.42,EUR,2025-04-28,410,2025,4,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2025/04 but IR posted in later period 2025/06. Exposure 696458.72 EUR requires period-end accrual adjustment.,POST
1010,Company Code DE 1010,4500100148,0050100006,66000000,631679.18,682213.51,EUR,2025-07-26,321,2025,7,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2025/07 but IR posted in later period 2025/09. Exposure 631679.18 EUR requires period-end accrual adjustment.,POST
1010,Company Code DE 1010,4500100086,0050100005,62000000,573069.46,618915.02,EUR,2025-12-07,187,2025,12,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2025/12 but IR posted in later period 2026/01. Exposure 573069.46 EUR requires period-end accrual adjustment.,POST
3010,Company Code AU 3010,4500100132,0050100005,63000000,505605.1,333699.37,AUD,2026-04-02,71,2026,4,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2026/04 but IR posted in later period 2026/06. Exposure 505605.1 AUD requires period-end accrual adjustment.,POST
5010,Company Code EE 5010,4500100101,0050100021,63000000,454322.6,490668.41,EUR,2026-01-02,161,2026,1,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2026/01 but IR posted in later period 2026/02. Exposure 454322.6 EUR requires period-end accrual adjustment.,POST
3010,Company Code AU 3010,4500100129,0050100015,64000000,415388.92,274156.69,AUD,2025-07-09,338,2025,7,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2025/07 but IR posted in later period 2025/09. Exposure 415388.92 AUD requires period-end accrual adjustment.,POST
4010,Company Code HR 4010,4500100196,0050100019,64000000,415231.28,448449.78,EUR,2025-09-03,282,2025,9,Price Variance,"Price Variance: Quantities match but billing variance of 415231.28 EUR detected (GR amount 669727.88 vs IR amount 254496.6). Likely cause: price change, surcharge, or invoicing error vs PO terms.",POST
2210,Company Code CH 2210,4500100028,0050100004,64000000,393605.62,440838.29,CHF,2026-01-21,142,2026,1,Cross-Period GR-IR,Cross-Period Cut-Off: GR posted in period 2026/01 but IR posted in later period 2026/02. Exposure 393605.62 CHF requires period-end accrual adjustment.,POST


In [0]:
# ── Step 2: Persist as Gold Delta Table ──────────────────────────────────────
df_fact.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"`{CATALOG}`.`{SCHEMA_GOLD}`.{TARGET_TABLE}")

print(f"✓ Saved Delta table: {CATALOG}.{SCHEMA_GOLD}.{TARGET_TABLE}")
print(f"  Rows: {spark.table(f'`{CATALOG}`.`{SCHEMA_GOLD}`.{TARGET_TABLE}').count()}")

✓ Saved Delta table: [pe]_xx102_pe_finance.gold.fact_accrual_worklist
  Rows: 87


In [0]:
# ── Step 3: Export as CSV to UC Volume ────────────────────────────────────────
# Convert to pandas and save as single CSV file
import os

csv_path = f"{OUTPUT_PATH}/fact_accrual_worklist.csv"
df_fact.toPandas().to_csv(csv_path, index=False)

print(f"✓ CSV exported to: {csv_path}")
print(f"  File size: {os.path.getsize(csv_path):,} bytes")

✓ CSV exported to: /Volumes/[pe]_xx102_pe_finance/gold/accrual_matching/output/fact_accrual_worklist.csv
  File size: 24,697 bytes


In [0]:
# ── Step 4: Verify Both Outputs ──────────────────────────────────────────────
print("═" * 70)
print("  GOLD LAYER VERIFICATION")
print("═" * 70)

# Verify Delta table
gold_df = spark.table(f"`{CATALOG}`.`{SCHEMA_GOLD}`.{TARGET_TABLE}")
print(f"\n  Delta Table: {CATALOG}.{SCHEMA_GOLD}.{TARGET_TABLE}")
print(f"  Rows: {gold_df.count()}")
print(f"  Columns: {gold_df.columns}")

# Verify CSV output (single file written by toPandas().to_csv())
import os
csv_file = f"{OUTPUT_PATH}/fact_accrual_worklist.csv"
try:
    csv_size = os.path.getsize(csv_file)
    print(f"\n  CSV Output: {csv_file}")
    print(f"  File size: {csv_size:,} bytes")
except FileNotFoundError:
    print(f"\n  CSV not found at: {csv_file}")

print(f"\n{'─' * 70}")
print("  Exposure by Company Code:")
gold_df.groupBy("CompanyCode", "CompanyCodeName").agg(
    F.count("*").alias("Items"),
    F.sum("Exposure_Amount").alias("Total_Exposure")
).orderBy("CompanyCode").show()

print("✓ Gold layer complete")

══════════════════════════════════════════════════════════════════════
  GOLD LAYER VERIFICATION
══════════════════════════════════════════════════════════════════════

  Delta Table: [pe]_xx102_pe_finance.gold.fact_accrual_worklist
  Rows: 87
  Columns: ['CompanyCode', 'CompanyCodeName', 'PurchaseOrder', 'Supplier', 'GLAccount', 'Exposure_Amount', 'Exposure_Amount_USD', 'CompanyCodeCurrency', 'Oldest_GR_Date', 'Days_Open', 'Fiscal_Year', 'Fiscal_Period', 'Worklist_Category', 'RCA', 'Recommended_Action']

  CSV Output: /Volumes/[pe]_xx102_pe_finance/gold/accrual_matching/output/fact_accrual_worklist.csv
  File size: 24,697 bytes

──────────────────────────────────────────────────────────────────────
  Exposure by Company Code:
+-----------+--------------------+-----+------------------+
|CompanyCode|     CompanyCodeName|Items|    Total_Exposure|
+-----------+--------------------+-----+------------------+
|       1010|Company Code DE 1010|   19|2519924.9800000004|
|       2210|Company Co